<a href="https://colab.research.google.com/github/nananti360/Data-Science-Projects/blob/main/Nana%20Nti%20Agyekum_Network%20Intrusion%20Detection%20System%20Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import cloudpickle  # Changed joblib to cloudpickle to match scoring notebook requirements
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

# 1. Saving Datas from Google Drive to Variables
train_path = '/content/drive/MyDrive/nids project dataset/student_train.csv'
test_path  = '/content/drive/MyDrive/nids project dataset/student_test.csv'

# 2. Load Datasets Into Dataframe
df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

print(f"Train Shape: {df_train.shape} | Test Shape: {df_test.shape}")

# 3. Clean infinity values & space around strings
for df in [df_train, df_test]:
    df.columns = df.columns.str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

# 3.5. Exclude Non-Feature Columns
target_col = df_train.columns[-1]

ID_COLS = ['source_ip_address', 'destination_ip_address', 'capture_sensor_id']
TIME_COLS = [
    'flow_start_timestamp_milliseconds',
    'flow_end_timestamp_milliseconds',
    'ip_time_to_live',
    'source_to_destination_inter_arrival_time_total_milliseconds',
    'destination_to_source_inter_arrival_time_total_milliseconds'
]

# Build list of non-feature columns to drop
non_feature_cols = ID_COLS + TIME_COLS + ['destination_port', target_col]
non_feature_cols = [c for c in non_feature_cols if c in df_train.columns]

# Select only valid features
feature_cols = [c for c in df_train.columns if c not in non_feature_cols]

# 4. Prepare Features & Binary Ground Truth
X_train = df_train[feature_cols].copy()
X_test  = df_test[feature_cols].copy()

# Convert multi-class labels into binary: 0 = Normal, 1 = Any Attack
raw_labels_test = df_test[target_col].astype(str)
normal_label = raw_labels_test.value_counts().index[0]
y_test_binary_true = np.where(raw_labels_test == normal_label, 0, 1)

# Encode remaining non-numeric feature columns
for col in X_train.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col]  = X_test[col].map(lambda s: '<unknown>' if s not in le.classes_ else s)
    le.classes_  = np.append(le.classes_, '<unknown>')
    X_test[col]  = le.transform(X_test[col].astype(str))

# 5, 5.5, & 6. Unified Unsupervised Pipeline (Scaler -> PCA -> Isolation Forest)
nids_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10, random_state=42)),
    ('model', IsolationForest(
        n_estimators=200,
        contamination=0.04,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the pipeline on feature columns
nids_pipeline.fit(X_train)

# 7. Extract Decision Scores
raw_scores = -nids_pipeline.decision_function(X_test)

normal_mask = (y_test_binary_true == 0)
normal_scores = raw_scores[normal_mask]
threshold_5pct = np.percentile(normal_scores, 95)

# Binary predictions mapped to 5% FPR cutoff
y_test_pred_5pct = np.where(raw_scores >= threshold_5pct, 1, 0)

# 8. Evaluation
print("\n--- TEAM 1 - NIDS PRELIMINARY MODEL ---")
print("\n--- Test Set Evaluation Report ---")
print(classification_report(y_test_binary_true, y_test_pred_5pct, target_names=['Normal (0)', 'Attack (1)'], zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test_binary_true, y_test_pred_5pct))

# Extract the trained IsolationForest model and the scaling/PCA transformer
detector = nids_pipeline.named_steps['model']

scaler_transformer = Pipeline([
    ('scaler', nids_pipeline.named_steps['scaler']),
    ('pca', nids_pipeline.named_steps['pca'])
])

# Save using cloudpickle as expected by load_model() in the scoring notebook
with open('/content/drive/MyDrive/nids_prelim_model.pkl', 'wb') as f:
    cloudpickle.dump(detector, f)

with open('/content/drive/MyDrive/nids_scaler.pkl', 'wb') as f:
    cloudpickle.dump(scaler_transformer, f)

with open('/content/drive/MyDrive/feature_cols.pkl', 'wb') as f:
    cloudpickle.dump(feature_cols, f)


Train Shape: (400000, 41) | Test Shape: (200000, 41)

--- TEAM 1 - NIDS PRELIMINARY MODEL ---

--- Test Set Evaluation Report ---
              precision    recall  f1-score   support

  Normal (0)       0.99      0.95      0.97    195939
  Attack (1)       0.11      0.31      0.17      4061

    accuracy                           0.94    200000
   macro avg       0.55      0.63      0.57    200000
weighted avg       0.97      0.94      0.95    200000

Confusion Matrix:
 [[186142   9797]
 [  2789   1272]]
